In [1]:
import os, json
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

stats_dir = 'stats'
pdb_dir = '/mnt/hdd/yenlin/data/Protein_Data_Bank'
pdb_download_dir = f'{pdb_dir}/pdb.gz'
plots_dir = 'plots'

multimericities = np.arange(2, 7)

# PARAMETERS
coverage_mode = 0
# similarity_cutoff = 1
similarity_cutoff = 0.95

# tag = f'sim_{similarity_cutoff}-PDB_ready'


In [2]:
### Pick GO terms for prediction

terms_of_interest = {
    # 'GO:0005488': 'binding', # 7935
    'GO:0036094': 'small molecule binding', # 5957
    'GO:0043167': 'ion binding', # 5253
    'GO:0043169': 'cation binding', # 3878
    'GO:0046872': 'metal ion binding', # 3841
    'GO:1901363': 'heterocyclic compound binding', # 2842
    'GO:1901265': 'nucleoside phosphate binding', # 2446
    'GO:0000166': 'nucleotide binding', # 2438
    'GO:0005515': 'protein binding', # 2310
    'GO:0043168': 'anion binding', # 2103
    'GO:0042802': 'identical protein binding', # 1579
    'GO:0017076': 'purine nucleotide binding', # 1570
    'GO:0097367': 'carbohydrate derivative binding', # 1476
    'GO:0003676': 'nucleic acid binding', # 1467
    'GO:0030554': 'adenyl nucleotide binding', # 1460
    'GO:0032553': 'ribonucleotide binding', # 1292
    'GO:0032555': 'purine ribonucleotide binding', # 1184
    'GO:0035639': 'purine ribonucleoside triphosphate binding', # 1130
    'GO:0032559': 'adenyl ribonucleotide binding', # 1070
    'GO:0005524': 'ATP binding', # 1029
    'GO:0046914': 'transition metal ion binding', # 1021
    # 'GO:0003677': 'DNA binding', # 971
    # 'GO:0008270': 'zinc ion binding', # 607
    # 'GO:0003723': 'RNA binding', # 561
    # 'GO:0000287': 'magnesium ion binding', # 536
    # 'GO:0019899': 'enzyme binding', # 466
    # 'GO:0003700': 'DNA-binding transcription factor activity', # 452
    # 'GO:0019842': 'vitamin binding', # 438
    # 'GO:0005102': 'signaling receptor binding', # 401
    # 'GO:0043565': 'sequence-specific DNA binding', # 401
    # 'GO:0003690': 'double-stranded DNA binding', # 380
    # 'GO:1990837': 'sequence-specific double-stranded DNA binding', # 341
    # 'GO:0044877': 'protein-containing complex binding', # 339
    # 'GO:0000976': 'transcription cis-regulatory region binding', # 332
    # 'GO:0001067': 'transcription regulatory region nucleic acid binding', # 332
    # 'GO:0008289': 'lipid binding', # 314
    # 'GO:0070279': 'vitamin B6 binding', # 308
    # 'GO:0030170': 'pyridoxal phosphate binding', # 307
    # 'GO:0030246': 'carbohydrate binding', # 278
    # 'GO:0051287': 'NAD binding', # 245
    # 'GO:0046906': 'tetrapyrrole binding', # 240
    # 'GO:0050661': 'NADP binding', # 225
    # 'GO:0020037': 'heme binding', # 221
    # 'GO:0019900': 'kinase binding', # 201
}


In [3]:
### READ GO ANNOTATIONS

with open(f'{stats_dir}/OUT-3-2.go_annotations.json', 'r') as f:
    go_annotations = json.load(f)

num2name_count = np.loadtxt(f'{stats_dir}/OUT-3-2.go_num2names_and_count.txt', dtype=np.str_, quotechar='"')

num2name = {num2name_count[i, 0]: num2name_count[i, 2] for i in range(num2name_count.shape[0])}

print(f'# entries with annotation: {len(go_annotations)}')
print(f'# GO types: {len(num2name)}')
print()


go_count_selected = {}

for term in terms_of_interest:
    go_count_selected[term] = 0

    for entry_id, term_list in go_annotations.items():
        if term in term_list:
            go_count_selected[term] += 1

for term, count in go_count_selected.items():
    print(f'{term} {count} "{num2name[term]}"')


# entries with annotation: 11359
# GO types: 17240

GO:0036094 5957 "small molecule binding"
GO:0043167 5253 "ion binding"
GO:0043169 3878 "cation binding"
GO:0046872 3841 "metal ion binding"
GO:1901363 2842 "heterocyclic compound binding"
GO:1901265 2446 "nucleoside phosphate binding"
GO:0000166 2438 "nucleotide binding"
GO:0005515 2310 "protein binding"
GO:0043168 2103 "anion binding"
GO:0042802 1579 "identical protein binding"
GO:0017076 1570 "purine nucleotide binding"
GO:0097367 1476 "carbohydrate derivative binding"
GO:0003676 1467 "nucleic acid binding"
GO:0030554 1460 "adenyl nucleotide binding"
GO:0032553 1292 "ribonucleotide binding"
GO:0032555 1184 "purine ribonucleotide binding"
GO:0035639 1130 "purine ribonucleoside triphosphate binding"
GO:0032559 1070 "adenyl ribonucleotide binding"
GO:0005524 1029 "ATP binding"
GO:0046914 1021 "transition metal ion binding"


In [4]:
### READ DATA FROM PREVIOUS SCRIPT

# META DATA
all_info = np.loadtxt(
    f'{stats_dir}/OUT-5.all_information-sim_{similarity_cutoff}-PDB_ready.csv',
    delimiter=',',
    dtype=np.str_,
    comments=None
)

header = all_info[0].tolist()
header[0] = header[0][2:]
all_info = all_info[1:]

print(header)

# FASTA
with open(
    f'{stats_dir}/OUT-5.all_sequences-sim_{similarity_cutoff}-PDB_ready.fasta',
    'r'
) as f:
    all_sequences = f.read().split('>')[1:]
all_sequences = np.array([
    entry.split('\n', 1)[1].replace('\n', '') for entry in all_sequences
])

print(all_sequences[1])

assert len(all_info) == len(all_sequences)
print()
print(len(all_info), 'entries')


['full_id', 'pdb_id', 'assembly_id', 'multimericity', 'auth_chain_id', 'asym_chain_id', 'seq_database', 'seq_database_accession']
AAAAANHSTQESGFDYEGLIDSELQKKRLDKSYRYFNNINRLAKEFPLAHRQREADKVTVWCSNDYLALSKHPEVLDAMHKTIDKYGCGAGGTRNIAGHNIPTLNLEAELATLHKKEGALVFSSCYVANDAVLSLLGQKMKDLVIFSDELNHASMIVGIKHANVKKHIFKHNDLNELEQLLQSYPKSVPKLIAFESVYSMAGSVADIEKICDLADKYGALTFLDEVHAVGLYGPHGAGVAEHCDFESHRASGIATPKTNDKGGAKTVMDRVDMITGTLGKSFGSVGGYVAASRKLIDWFRSFAPGFIFTTTLPPSVMAGATAAIRYQRCHIDLRTSQQKHTMYVKKAFHELGIPVIPNPSHIVPVLIGNADLAKQASDILINKHQIYVQAINFPTVARGTERLRITPTPGHTNDLSDILINAVDDVFNELQLPRVRDWESQGGLLGVGESGFVEESNLWTSSQLSLTNDDLNPNVRDPIVKQLEVSSGIKQ

11043 entries


In [5]:
### Build labels for prediction

go_labels = []
for full_id in all_info[:,header.index('full_id')]:
    entry_terms = go_annotations[full_id]
    entry_labels = []
    for term in terms_of_interest:
        entry_labels.append(1 if term in entry_terms else 0)
    go_labels.append(entry_labels)

go_labels = np.array(go_labels)
print(go_labels.shape)

print(f'# entries without labels: {np.sum(np.sum(go_labels, axis=1) == 0)}')


(11043, 20)
# entries without labels: 3528


In [6]:
### SAVE

np.savetxt(
    f'{stats_dir}/OUT-6.go_annotations_of_interest.csv',
    [
        [term, count, num2name[term]]
        for term, count in go_count_selected.items()
    ],
    fmt='%s,%s,"%s"'
)

np.savetxt(
    f'{stats_dir}/OUT-6.labels_for_prediction.csv',
    go_labels,
    delimiter=',',
    fmt='%d'
)

np.savetxt(
    f'{stats_dir}/OUT-6.entries_id.txt',
    all_info[:,header.index('full_id')],
    fmt='%s'
)
